<a href="https://colab.research.google.com/github/lejan392/case_study-data_acquisition/blob/main/case_study_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
population_df=pd.read_csv('/content/drive/MyDrive/DSA B13/Dataset/population.csv')
print(population_df.head(5))
print(population_df.shape)
print(population_df.dtypes)


         country  population
0  United States   331002651
1          China  1411778724
2          India  1380004385
3          Japan   125836021
4        Germany    83240525
(25, 2)
country       object
population     int64
dtype: object


In [3]:
gdp_df=pd.read_excel('/content/drive/MyDrive/DSA B13/Dataset/gdp.xlsx')
print(gdp_df.head())
print(gdp_df.columns)
print(gdp_df.dtypes)

         Country       GDP
0  United States  30500000
1          China  19230000
2          India   4187000
3          Japan   4180000
4        Germany   4744000
Index(['Country', 'GDP'], dtype='object')
Country    object
GDP         int64
dtype: object


In [4]:
internet_df=pd.read_json('/content/drive/MyDrive/DSA B13/Dataset/internet_users.json')
internet_df.head()
internet_df=pd.DataFrame(internet_df)
print(internet_df.head())
print(internet_df.dtypes)

         country  internet_users
0  United States       312000000
1          China      1080000000
2          India       850000000
3          Japan       118000000
4        Germany        79000000
country           object
internet_users     int64
dtype: object


In [5]:
literacy_df = pd.read_xml("/content/drive/MyDrive/DSA B13/Dataset/literacy_rate.xml",xpath=".//country")
literacy_df=literacy_df.rename(columns={"name": "country"})

print(literacy_df.head())
print(literacy_df.dtypes)

         country  literacy_rate
0  United States           99.0
1          China           97.2
2          India           77.7
3          Japan           99.0
4        Germany           99.0
country           object
literacy_rate    float64
dtype: object


In [6]:
datasets = {
    "Population": population_df,
    "GDP": gdp_df,
    "Internet Users": internet_df,
    "Literacy Rate": literacy_df
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Rows and columns:", df.shape)
    print("Column names:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())
    print("Duplicate records:", df.duplicated().sum())


Population
Rows and columns: (25, 2)
Column names: ['country', 'population']

Data types:
country       object
population     int64
dtype: object

Missing values:
country       0
population    0
dtype: int64
Duplicate records: 0

GDP
Rows and columns: (25, 2)
Column names: ['Country', 'GDP']

Data types:
Country    object
GDP         int64
dtype: object

Missing values:
Country    0
GDP        0
dtype: int64
Duplicate records: 0

Internet Users
Rows and columns: (25, 2)
Column names: ['country', 'internet_users']

Data types:
country           object
internet_users     int64
dtype: object

Missing values:
country           0
internet_users    0
dtype: int64
Duplicate records: 0

Literacy Rate
Rows and columns: (25, 2)
Column names: ['country', 'literacy_rate']

Data types:
country           object
literacy_rate    float64
dtype: object

Missing values:
country          0
literacy_rate    0
dtype: int64
Duplicate records: 0


In [7]:
gdp_df = gdp_df.rename(columns={"Country": "country"})
gdp_df

,country,GDP
0,United States,30500000
1,China,19230000
2,India,4187000
3,Japan,4180000
4,Germany,4744000
5,United Kingdom,3839000
6,France,3211000
7,Italy,2420000
8,Canada,2250000
9,Australia,1880000


In [13]:
population_df["population"] = pd.to_numeric(population_df["population"])
gdp_df["GDP"] = pd.to_numeric(gdp_df["GDP"])
internet_df["internet_users"] = pd.to_numeric(internet_df["internet_users"])
literacy_df["literacy_rate"] = pd.to_numeric(literacy_df["literacy_rate"])
print(population_df.dtypes)
print(gdp_df.dtypes)
print(internet_df.dtypes)
print(literacy_df.dtypes)


country       object
population     int64
dtype: object
country    object
GDP         int64
dtype: object
country           object
internet_users     int64
dtype: object
country           object
literacy_rate    float64
dtype: object


In [9]:
merged_df = (population_df
    .merge(gdp_df, on="country", how="inner")
    .merge(internet_df, on="country", how="inner")
    .merge(literacy_df, on="country", how="inner")
)

print(merged_df.head())

         country  population       GDP  internet_users  literacy_rate
0  United States   331002651  30500000       312000000           99.0
1          China  1411778724  19230000      1080000000           97.2
2          India  1380004385   4187000       850000000           77.7
3          Japan   125836021   4180000       118000000           99.0
4        Germany    83240525   4744000        79000000           99.0


In [16]:
merged_df["Internet Penetration (%)"] = (
    merged_df["internet_users"] / merged_df["population"] * 100
).round(2)

merged_df["GDP Per Capita"] = (
    merged_df["GDP"] * 1_000_000 / merged_df["population"]
).round(2)

print(merged_df.head())

         country  population       GDP  internet_users  literacy_rate  \
0  United States   331002651  30500000       312000000           99.0   
1          China  1411778724  19230000      1080000000           97.2   
2          India  1380004385   4187000       850000000           77.7   
3          Japan   125836021   4180000       118000000           99.0   
4        Germany    83240525   4744000        79000000           99.0   

   Internet Penetration (%)  GDP Per Capita  
0                     94.26        92144.28  
1                     76.50        13621.11  
2                     61.59         3034.05  
3                     93.77        33217.83  
4                     94.91        56991.47  


In [15]:
import sqlite3

conn = sqlite3.connect("country_statistics.db")

merged_df.to_sql(
    "country_statistics",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

In [17]:
conn = sqlite3.connect("country_statistics.db")

check_df = pd.read_sql_query(
    "SELECT * FROM country_statistics LIMIT 5;",
    conn
)

print(check_df)

conn.close()

         country  population       GDP  internet_users  literacy_rate  \
0  United States   331002651  30500000       312000000           99.0   
1          China  1411778724  19230000      1080000000           97.2   
2          India  1380004385   4187000       850000000           77.7   
3          Japan   125836021   4180000       118000000           99.0   
4        Germany    83240525   4744000        79000000           99.0   

   Internet Penetration (%)  GDP Per Capita  
0                     94.26        92144.28  
1                     76.50        13621.11  
2                     61.59         3034.05  
3                     93.77        33217.83  
4                     94.91        56991.47  


In [19]:
# 1.
print(merged_df.nlargest(5, "GDP")[["country", "GDP"]])

# 2.
print(merged_df.nlargest(5, "population")[["country", "population"]])

# 3.
print(merged_df["literacy_rate"].mean())

# 4.
print(merged_df[merged_df["literacy_rate"] > 90][["country", "literacy_rate"]])

# 5.
print(merged_df[merged_df["Internet Penetration (%)"] > 70][
    ["country", "Internet Penetration (%)"]
])

# 6.
print(merged_df.nlargest(5, "GDP Per Capita")[["country", "GDP Per Capita"]])

# 7.
print(merged_df["population"].sum())

# 8.
print(merged_df["Internet Penetration (%)"].mean())

# 9.
print(merged_df.loc[
    merged_df["internet_users"].idxmax(),
    ["country", "internet_users"]
])

# 10.
print(merged_df["literacy_rate"].corr(merged_df["Internet Penetration (%)"]))

         country       GDP
0  United States  30500000
1          China  19230000
4        Germany   4744000
2          India   4187000
3          Japan   4180000
          country  population
1           China  1411778724
2           India  1380004385
0   United States   331002651
15      Indonesia   273523615
10         Brazil   212559417
97.248
           country  literacy_rate
0    United States           99.0
1            China           97.2
3            Japan           99.0
4          Germany           99.0
5   United Kingdom           99.0
6           France           99.0
7            Italy           99.0
8           Canada           99.0
9        Australia           99.0
10          Brazil           93.2
11          Russia           99.7
12     South Korea           98.8
13           Spain           98.6
14          Mexico           95.4
15       Indonesia           96.0
16    Saudi Arabia           97.3
17          Turkey           96.7
18    South Africa           95.0
19   

In [21]:
conn = sqlite3.connect("country_statistics.db")

# 11.
print(pd.read_sql_query("""
    SELECT country, GDP
    FROM country_statistics
    ORDER BY GDP DESC
    LIMIT 5;
""", conn))

# 12.
print(pd.read_sql_query("""
    SELECT AVG(literacy_rate) AS average_literacy_rate
    FROM country_statistics;
""", conn))

# 13.
print(pd.read_sql_query("""
    SELECT country, literacy_rate
    FROM country_statistics
    WHERE literacy_rate > 90;
""", conn))

# 14.
print(pd.read_sql_query("""
    SELECT country, "Internet Penetration (%)"
    FROM country_statistics
    ORDER BY "Internet Penetration (%)" DESC
    LIMIT 5;
""", conn))

# 15.
print(pd.read_sql_query("""
    SELECT SUM(population) AS total_population
    FROM country_statistics;
""", conn))

# 16.
print(pd.read_sql_query("""
    SELECT country, "GDP Per Capita"
    FROM country_statistics
    ORDER BY "GDP Per Capita" DESC
    LIMIT 1;
""", conn))

# 17.
merged_df.to_csv("country_statistics.csv", index=False)

# 18.
print(pd.read_sql_query("""
    SELECT country, "GDP Per Capita"
    FROM country_statistics
    ORDER BY "GDP Per Capita" DESC
    LIMIT 10;
""", conn))

conn.close()

         country       GDP
0  United States  30500000
1          China  19230000
2        Germany   4744000
3          India   4187000
4          Japan   4180000
   average_literacy_rate
0                 97.248
           country  literacy_rate
0    United States           99.0
1            China           97.2
2            Japan           99.0
3          Germany           99.0
4   United Kingdom           99.0
5           France           99.0
6            Italy           99.0
7           Canada           99.0
8        Australia           99.0
9           Brazil           93.2
10          Russia           99.7
11     South Korea           98.8
12           Spain           98.6
13          Mexico           95.4
14       Indonesia           96.0
15    Saudi Arabia           97.3
16          Turkey           96.7
17    South Africa           95.0
18       Argentina           99.1
19     Netherlands           99.0
20     Switzerland           99.0
21          Sweden           99.0
22    